# Computational Estimation of Protein–Protein Binding Affinity

---

## 1. Introduction

Protein–protein interactions (PPIs) are fundamental to virtually every biological process, from signal transduction and immune recognition to enzyme catalysis and gene regulation. Understanding **how tightly two proteins bind** — and *why* — is therefore central to structural biology, drug discovery, and protein engineering.

The thermodynamic quantity that governs whether a binding event is spontaneous and how strong that interaction is, is the **Gibbs free energy of binding**, $\Delta G_\text{bind}$. A negative value of $\Delta G_\text{bind}$ indicates a thermodynamically favourable, spontaneous association:

$$
\Delta G_\text{bind} = \Delta H_\text{bind} - T\,\Delta S_\text{bind}
$$

where $\Delta H_\text{bind}$ is the **enthalpy** of binding (dominated by non-covalent interactions at the interface), $T$ is the absolute temperature (here 298 K), and $\Delta S_\text{bind}$ is the **entropy** change upon complex formation. Both terms are expressed in **kJ mol⁻¹**, and $\Delta S$ in **kJ mol⁻¹ K⁻¹**.

Rather than relying on computationally expensive methods such as free-energy perturbation (FEP) or MM/PBSA, this project implements a **physics-based, additive scoring function** that decomposes $\Delta G_\text{bind}$ into chemically interpretable contributions. Each term is estimated from the three-dimensional structure of the protein–protein complex.

---

## 2. Enthalpic Contributions — $\Delta H_\text{bind}$

The enthalpic component captures the **direct non-covalent interactions** formed between the two protein chains at their interface. Six interaction types are considered:

### 2.1 Hydrogen Bonds

Hydrogen bonds (H-bonds) arise when a hydrogen atom covalently bound to an electronegative donor (N, O, or S) interacts with a lone pair on an electronegative acceptor. At protein–protein interfaces, H-bonds are formed between backbone amide/carbonyl groups and between polar side chains (Ser, Thr, Tyr, Asn, Gln, His, Arg, Lys, Asp, Glu).

Each H-bond contributes approximately **−2 to −8 kJ mol⁻¹** depending on geometry (donor–acceptor distance and D–H···A angle). In our model, H-bonds are identified by distance and angular cutoffs, and each is assigned a fixed average energy.

### 2.2 Salt Bridges

Salt bridges are electrostatic interactions between oppositely charged residues (Arg, Lys ↔ Asp, Glu). They are among the strongest directional interactions at protein interfaces, contributing **−10 to −20 kJ mol⁻¹** per pair when fully buried, though their strength is significantly attenuated by solvent screening at exposed surfaces.

Detection is based on the distance between charged heavy atoms (typically < 4 Å).

### 2.3 Hydrophobic Contacts

Non-polar residues (Ala, Val, Leu, Ile, Met, Phe, Trp, Pro) tend to cluster at protein cores and interfaces, driven by the **hydrophobic effect** — the thermodynamically favourable burial of apolar surface away from water. In the enthalpic model, hydrophobic contacts between apolar carbon atoms of opposing chains within a cutoff distance are summed, each contributing a small stabilising energy per contact.

### 2.4 π–π Stacking

Aromatic residues (Phe, Tyr, Trp, His) can interact through **π-electron cloud overlap** in two geometries:
- **Face-to-face (sandwich)**: parallel stacking, ~−10 kJ mol⁻¹  
- **Edge-to-face (T-shaped)**: one ring perpendicular to the other, ~−6 kJ mol⁻¹

These interactions are identified by the distance between ring centroids and the dihedral angle between ring planes.

### 2.5 Dipole–Dipole Interactions

Polar bonds (C=O, N–H, C–F, etc.) carry partial charges that give rise to **permanent electric dipoles**. When two such dipoles are in close proximity and favourable orientation (head-to-tail or antiparallel), they stabilise the complex. This term captures medium-range electrostatic effects not classified as H-bonds or salt bridges, and is estimated from the partial charges and geometry of interacting polar groups.

### 2.6 Halogen Bonds

Halogen bonds (X-bonds) occur when a halogen substituent (Cl, Br, I) acts as an electrophilic **σ-hole** donor interacting with a Lewis base (carbonyl oxygen, amine nitrogen). Although rare in natural protein–protein interfaces, they can be present in engineered proteins or when halogenated ligands or cofactors are involved. They are directional (optimal at ~180° C–X···O angle) and contribute **−4 to −12 kJ mol⁻¹**.

### Total Enthalpy

$$
\Delta H_\text{bind} = E_\text{H-bond} + E_\text{salt} + E_\text{hydrophobic} + E_{\pi\text{-}\pi} + E_\text{dipole} + E_\text{halogen}
$$

---

## 3. Entropic Contributions — $\Delta S_\text{bind}$

Entropy changes upon binding are typically **unfavourable** (negative $\Delta S$, raising $\Delta G$) because complex formation restricts the conformational and translational freedom of both partners. Five entropic terms are computed:

### 3.1 Translational & Rotational Entropy

When two free proteins associate into a single complex, they lose **three translational** and **three rotational** degrees of freedom (one set per protein is frozen). This is the single largest unfavourable entropic cost of binding, estimated using the Sackur–Tetrode equation and rigid-rotor approximations:

$$
-T\Delta S_\text{trans/rot} \approx +20\text{ to }+60\;\text{kJ mol}^{-1}
$$

This penalty is largely **independent of interface size** and is a fundamental thermodynamic cost of bimolecular association.

### 3.2 Backbone Conformational Entropy

Residues at or near the binding interface often become **conformationally restricted** upon complex formation — their backbone $\phi$/$\psi$ dihedral angles explore a narrower region of the Ramachandran map. This is quantified by the change in backbone dihedral entropy, estimated from B-factors or normal-mode analysis. The penalty scales with the number of interface residues that become rigidified.

### 3.3 Side-Chain Conformational Entropy

Side chains lose rotameric freedom upon binding: a flexible Lys or Arg in solution may sample many $\chi$-angle combinations, but in the complex it is often locked into a single rotamer to optimise a salt bridge or H-bond. The side-chain entropy loss is estimated per rotatable bond that becomes restricted:

$$
\Delta S_\text{sidechain} = -k_B \sum_i \ln\left(\frac{\Omega_i^\text{bound}}{\Omega_i^\text{free}}\right)
$$

where $\Omega_i$ is the number of accessible rotameric states for bond $i$.

### 3.4 Hydrophobic (Solvation) Entropy

Burial of **non-polar surface area** (SASA) upon binding releases ordered water molecules from hydrophobic surfaces back into bulk solvent. This is **entropically favourable** (positive contribution to $\Delta S$) and is the principal driving force of the classical hydrophobic effect:

$$
\Delta S_\text{hydrophobic} = \sigma_\text{np} \cdot \Delta\text{SASA}_\text{np}
$$

where $\sigma_\text{np}$ is an empirical coefficient (~0.1 kJ mol⁻¹ Å⁻²) and $\Delta\text{SASA}_\text{np}$ is the change in non-polar solvent-accessible surface area.

### 3.5 Solvent–Solute Entropy

The solvent–solute entropy term captures the reorganisation of water molecules at the protein surface upon complex formation. Following the framework of Sun (2022), water at a protein interface is partitioned into two populations with distinct hydrogen-bond geometries:

- **Interfacial water** — the topmost hydration layer, where the protein surface disrupts the bulk tetrahedral network. These molecules form weaker DA (single-donor/single-acceptor) hydrogen bonds, giving them higher configurational entropy than bulk water.
- **Bulk water** — retains the full tetrahedral DDAA (double-donor/double-acceptor) network; it is more ordered and carries lower entropy.

A **critical radius** $R_c$ separates two solvation regimes:

$$
R_c = \frac{8\,|\Delta G_\text{DDAA}|\, r_{\text{H}_2\text{O}}}{|\Delta G_\text{water-water}|}
$$

For proteins (large solutes with $R \gg R_c$), the bulk ordering term dominates, giving an overall **unfavourable** (negative) solvent–solute entropy contribution that opposes binding. The total solvent–solute entropy is the sum of the interfacial and bulk contributions:

$$
\Delta S_\text{solvent-solute} = \Delta S_\text{interfacial} + \Delta S_\text{bulk}
$$

$$
\Delta S_\text{interfacial} = \frac{\Delta H_\text{DDAA}}{T} \cdot \frac{4\, r_{\text{H}_2\text{O}}}{R}, \qquad
\Delta S_\text{bulk} = -\Delta S_\text{DDAA} \cdot \left(1 - \frac{4\, r_{\text{H}_2\text{O}}}{R}\right)
$$

where $R$ is the sphere-equivalent radius of the protein, $r_{\text{H}_2\text{O}} = 1.9$ Å is the effective water molecule radius, and $\Delta H_\text{DDAA}$ and $\Delta S_\text{DDAA}$ are the van't Hoff enthalpy and entropy of the tetrahedral H-bond network (Sun 2022, Fig. 5).

### Total Entropy

$$
\Delta S_\text{bind} = \Delta S_\text{trans/rot} + \Delta S_\text{backbone} + \Delta S_\text{sidechain} + \Delta S_\text{hydrophobic} + \Delta S_\text{solvent-solute}
$$

---

## 4. The Binding Free Energy

Combining all enthalpic and entropic contributions, the estimated Gibbs free energy of binding is:

$$
\Delta G_\text{bind} = \underbrace{\left(E_\text{H-bond} + E_\text{salt} + E_\text{hydrophobic} + E_{\pi\text{-}\pi} + E_\text{dipole} + E_\text{halogen}\right)}_{\Delta H_\text{bind}}
- T\underbrace{\left(\Delta S_\text{trans/rot} + \Delta S_\text{backbone} + \Delta S_\text{sidechain} + \Delta S_\text{hydrophobic} + \Delta S_\text{solvent-solute}\right)}_{\Delta S_\text{bind}}
$$

All energies are expressed in **kJ mol⁻¹**. A more negative $\Delta G_\text{bind}$ indicates a tighter, more stable complex. As a reference, experimentally measured PPIs range from weak transient interactions ($\Delta G \approx -20$ kJ mol⁻¹) to ultra-tight complexes such as barnase–barstar ($\Delta G \approx -75$ kJ mol⁻¹).

> **Note on model limitations**: This scoring function is additive and uses fixed per-contact energy parameters. It does not account for allosteric effects, induced fit, or solvent screening in atomic detail. It is best interpreted as a comparative tool across a set of structures rather than an absolute predictor of experimental $K_d$ values.

---

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt